# TextCNN 卷积核与稳定性实验

固定数据处理、模型其他参数和训练方式，先比较卷积核宽度 2、3、4，再对候选配置和原配置进行多随机种子复验。每次按验证损失选择最佳轮次；测试集不参与实验。


In [ ]:
# 加载数据和实验依赖
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from sentiment_analysis.evaluation import classification_metrics
from sentiment_analysis.paths import CHECKPOINTS_DIR, CLEANED_DIR, TABLES_DIR
from sentiment_analysis.sequence_data import build_vocabulary, SentimentDataset, TOKEN_PATTERN
from sentiment_analysis.textcnn import TextCNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_data = pd.read_csv(CLEANED_DIR / "train.csv")
validation_data = pd.read_csv(CLEANED_DIR / "val.csv")

max_words = 20_000
max_length = 512
batch_size = 32
max_epochs = 20
patience = 4
learning_rate = 0.001

vocabulary = build_vocabulary(train_data["text"], max_words=max_words)
train_dataset = SentimentDataset(train_data, vocabulary, max_length)
validation_dataset = SentimentDataset(validation_data, vocabulary, max_length)
validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {device}")
print(f"Train: {len(train_dataset):,} | Validation: {len(validation_dataset):,}")
print(f"Vocabulary: {len(vocabulary):,}")


In [ ]:
# 训练与验证
def train_one_epoch(model, data_loader, optimizer, loss_function):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for token_ids, lengths, labels in data_loader:
        token_ids = token_ids.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(token_ids, lengths)
        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate_model(model, data_loader, loss_function):
    model.eval()
    total_loss = 0.0
    total = 0
    y_true = []
    y_prediction = []

    with torch.no_grad():
        for token_ids, lengths, labels in data_loader:
            token_ids = token_ids.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)

            logits = model(token_ids, lengths)
            loss = loss_function(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total += labels.size(0)
            y_true.extend(labels.cpu().tolist())
            y_prediction.extend(logits.argmax(dim=1).cpu().tolist())

    return total_loss / total, classification_metrics(y_true, y_prediction)


In [ ]:
# 按同一规则训练单组配置，并保存验证损失最低的模型
def run_experiment(kernel_size, random_seed):
    torch.manual_seed(random_seed)
    generator = torch.Generator().manual_seed(random_seed)
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, generator=generator,
    )

    model_config = {
        "vocabulary_size": len(vocabulary),
        "embedding_dim": 128,
        "num_filters": 64,
        "kernel_size": kernel_size,
        "padding_index": vocabulary["<PAD>"],
        "dropout": 0.5,
    }
    model = TextCNN(**model_config).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()

    best_validation_loss = float("inf")
    epochs_without_improvement = 0
    history = []
    checkpoint_path = CHECKPOINTS_DIR / f"textcnn_kernel{kernel_size}_seed{random_seed}.pt"

    for epoch in range(1, max_epochs + 1):
        train_loss, train_accuracy = train_one_epoch(model, train_loader, optimizer, loss_function)
        validation_loss, metrics = evaluate_model(model, validation_loader, loss_function)

        history.append({
            "KERNEL SIZE": kernel_size, "SEED": random_seed, "EPOCH": epoch,
            "TRAIN LOSS": train_loss, "TRAIN ACCURACY": train_accuracy,
            "VALIDATION LOSS": validation_loss, "VALIDATION ACCURACY": metrics["ACCURACY"],
            "VALIDATION F1": metrics["F1"],
        })
        print(
            f"Kernel {kernel_size} | Seed {random_seed} | Epoch {epoch} | "
            f"Train loss {train_loss:.4f} | Validation loss {validation_loss:.4f} | "
            f"Validation F1 {metrics['F1']:.4f}",
            flush=True,
        )

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            epochs_without_improvement = 0
            best_result = {
                "KERNEL SIZE": kernel_size, "SEED": random_seed, "BEST EPOCH": epoch,
                "VALIDATION LOSS": validation_loss, **metrics,
            }
            torch.save({
                "model_state_dict": model.state_dict(),
                "model_config": model_config,
                "vocabulary": vocabulary,
                "max_length": max_length,
                "token_pattern": TOKEN_PATTERN.pattern,
                "lowercase": True,
                "random_seed": random_seed,
                "epoch": epoch,
                "validation_loss": validation_loss,
            }, checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print("Early stopping.", flush=True)
                break

    return best_result, history


## 第一组：卷积核宽度比较

复用此前宽度为 3、随机种子为 42 的结果，训练宽度为 2 和 4 的模型。其他配置保持一致。


In [ ]:
# 读取此前的宽度 3 基线，并确认词表和模型配置
baseline_checkpoint = torch.load(
    CHECKPOINTS_DIR / "textcnn_extended_best.pt", map_location=device, weights_only=True,
)
assert baseline_checkpoint["vocabulary"] == vocabulary
assert baseline_checkpoint["token_pattern"] == TOKEN_PATTERN.pattern
assert baseline_checkpoint["max_length"] == max_length
assert baseline_checkpoint["model_config"]["kernel_size"] == 3
assert baseline_checkpoint["random_seed"] == 42

baseline_model = TextCNN(**baseline_checkpoint["model_config"]).to(device)
baseline_model.load_state_dict(baseline_checkpoint["model_state_dict"])
baseline_loss, baseline_metrics = evaluate_model(baseline_model, validation_loader, nn.CrossEntropyLoss())

baseline_result = {
    "KERNEL SIZE": 3,
    "SEED": 42,
    "BEST EPOCH": baseline_checkpoint["epoch"],
    "VALIDATION LOSS": baseline_loss,
    **baseline_metrics,
}
experiment_results = [baseline_result]
experiment_history = []

print("Existing baseline:", baseline_result)


In [ ]:
# 测试卷积核宽度 2 和 4
for kernel_size in [2, 4]:
    result, history = run_experiment(kernel_size, random_seed=42)
    experiment_results.append(result)
    experiment_history.extend(history)

    # 每完成一组就保存结果，方便检查
    pd.DataFrame(experiment_results).to_csv(TABLES_DIR / "textcnn_kernel_results.csv", index=False)
    pd.DataFrame(experiment_history).to_csv(TABLES_DIR / "textcnn_kernel_training_history.csv", index=False)

kernel_comparison = pd.DataFrame(experiment_results).sort_values("KERNEL SIZE")
display(kernel_comparison.round(4))


## 第二组：随机种子复验

按第一组的验证集 F1 选择候选卷积核宽度，再使用相同的两个新种子复验候选配置及原宽度 3 配置。每次仍按验证损失选取最佳轮次。


In [ ]:
# 对候选配置和原配置使用相同的额外随机种子
candidate_kernel = int(kernel_comparison.loc[kernel_comparison["F1"].idxmax(), "KERNEL SIZE"])
repeat_kernels = sorted({3, candidate_kernel})
print(f"Candidate kernel: {candidate_kernel} | Repeated kernels: {repeat_kernels}", flush=True)

for kernel_size in repeat_kernels:
    for random_seed in [123, 2026]:
        result, history = run_experiment(kernel_size, random_seed)
        experiment_results.append(result)
        experiment_history.extend(history)

        pd.DataFrame(experiment_results).to_csv(TABLES_DIR / "textcnn_kernel_results.csv", index=False)
        pd.DataFrame(experiment_history).to_csv(TABLES_DIR / "textcnn_kernel_training_history.csv", index=False)

all_results = pd.DataFrame(experiment_results)
display(all_results.sort_values(["KERNEL SIZE", "SEED"]).round(4))


In [ ]:
# 汇总相同随机种子下的候选配置
stability_results = all_results[all_results["KERNEL SIZE"].isin(repeat_kernels)]
stability_summary = (
    stability_results.groupby("KERNEL SIZE")
    .agg(
        RUNS=("F1", "count"),
        F1_MEAN=("F1", "mean"),
        F1_STD=("F1", "std"),
        F1_MIN=("F1", "min"),
        F1_MAX=("F1", "max"),
        ACCURACY_MEAN=("ACCURACY", "mean"),
    )
    .reset_index()
)

display(stability_summary.round(4))
stability_summary.to_csv(TABLES_DIR / "textcnn_kernel_stability.csv", index=False)


In [ ]:
# 展示卷积核比较与随机种子复验结果
import matplotlib.pyplot as plt
from sentiment_analysis.paths import FIGURES_DIR

figure, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(kernel_comparison["KERNEL SIZE"].astype(str), kernel_comparison["F1"])
axes[0].set_title("Kernel width comparison (seed 42)")
axes[0].set_xlabel("Kernel width")
axes[0].set_ylabel("Validation F1")
axes[0].set_ylim(0, 1)
for bar in axes[0].containers:
    axes[0].bar_label(bar, fmt="%.3f", padding=3)

for kernel_size, data in stability_results.groupby("KERNEL SIZE"):
    axes[1].plot(data["SEED"].astype(str), data["F1"], marker="o", linestyle="none", label=f"Kernel {kernel_size}")
axes[1].set_title("Repeated seeds")
axes[1].set_xlabel("Random seed")
axes[1].set_ylabel("Validation F1")
axes[1].set_ylim(0, 1)
axes[1].legend()

figure.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure.savefig(FIGURES_DIR / "textcnn_kernel_comparison.png", dpi=150)
plt.show()

## 阶段性结论

在种子 42 下，卷积核宽度 3 的验证 F1（0.8686）略高于宽度 2（0.8656）和宽度 4（0.8641）。宽度 3 在种子 42、123、2026 下的验证 F1 分别为 0.8686、0.8570、0.8682，均值约为 0.8646。

暂以宽度 3、种子 42、验证损失最低的第 6 轮模型作为 TextCNN 基线。种子 42 是最初使用的配置；多种子结果用于观察波动，而不是从中挑选最高分。

这只是有限的配置和种子比较，宽度 2 和 4 尚未进行多种子复验。所有结果均来自用于模型选择的同一验证集，测试集仍未使用。宽度 3、种子 42 的逐轮记录保存在 `textcnn_training_history.csv`；本 Notebook 的逐轮记录保存了其他新训练的运行。